In [ ]:
print(1)

In [ ]:

import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime
import pandas as pd
import numpy as np


In [ ]:
startdate = datetime(2021, 1, 1).timestamp()
enddate = datetime(2026, 4, 9).timestamp()

df_list = []

for ticker in ['AAPL', 'XOM', 'TSLA', 'WMT', 'PFE', 'NFLX', 'JPM', 'CAT']:
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)

    url = f'https://finance.yahoo.com/quote/{ticker}/history/?period1={startdate}&period2={enddate}'
    driver.get(url)

    try:
        accept = WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept all')]")))
        accept.click()
    except:
        pass
    
    if ticker == 'AAPL':
        headers = driver.find_element(By.XPATH, '//*[@id="main-content-wrapper"]/div[1]/div[3]/table/thead/tr')
        t = headers.text.split('\n')
        cols = t[0].split() + t[1:]
        cols.insert(0, 'ticker')

    table = driver.find_element(By.XPATH, '//*[@id="main-content-wrapper"]/div[1]/div[3]/table/tbody')

    for i in table.text.split('\n'):
        spl = i.split()

        if 'Dividend' in spl  or 'Stock' in spl:
            continue

        d = list(spl)[:3]
        d[0] = f'{d[0]} {d[1]} {d[2]}'
        d.pop(1)
        d.pop(1)

        val = list(map(float, spl[3:-1]))
        volume = float(spl[-1].replace(',', ''))

        d.extend(val)
        d.append(volume)
        d.insert(0, ticker)
        df_list.append(d)

    driver.quit()
    time.sleep(1)

    
df = pd.DataFrame(df_list, columns=cols)

df

In [ ]:
df.to_csv('quotes_all.csv')

In [ ]:
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

for ticker in ['AAPL']:
    startdate = datetime(2021, 1, 1).timestamp()
    enddate = datetime(2026, 4, 9).timestamp()

    url = f'https://finance.yahoo.com/quote/{ticker}/history/?period1={startdate}&period2={enddate}'
    driver.get(url)

    try:
        agree_button = WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept all')]")))
        agree_button.click()
    except:
        pass

    print(driver.title)
    



In [ ]:
headers = driver.find_element(By.XPATH, '//*[@id="main-content-wrapper"]/div[1]/div[3]/table/thead/tr')
headers.text

In [ ]:
t = headers.text.split('\n')
cols = t[0].split() + t[1:]
cols

In [ ]:
table = driver.find_element(By.XPATH, '//*[@id="main-content-wrapper"]/div[1]/div[3]/table/tbody')
table.text

In [ ]:
df_list = []
for i in table.text.split('\n'):
    spl = i.split()
    if 'Dividend' in spl:
        continue
    d = list(spl)[:3]
    d[0] = f'{d[0]} {d[1]} {d[2]}'
    d.pop(1)
    d.pop(1)

    val = list(map(float, spl[3:-1]))
    volume = float(spl[-1].replace(',', ''))

    d.extend(val)
    d.append(volume)

    df_list.append(d)

df_list

In [ ]:
df = pd.DataFrame(df_list, columns=cols)
df

In [ ]:

driver.quit()